# Known landmark localization in ℝ³

This example demonstrates 3D localization of a single point state `wX` from four known landmarks using `KnownLandmarkSquaredRangeFactor3`.

A single unknown point $x \in \mathbb{R}^3$ is observed through range measurements to several fixed, known landmarks $L_i$. A conventional range residual has the form

$$
e_i(x) = \|x - L_i\| - d_i,
$$

where $d_i$ is the measured range to landmark $L_i$. Because the Euclidean norm contains a square root, this residual is not polynomial in the state.

`KnownLandmarkSquaredRangeFactor2` and `KnownLandmarkSquaredRangeFactor3` instead use the squared-range residual

$$
e_i(x) = \|x - L_i\|^2 - d_i^2,
$$

which is quadratic in $x$. This polynomial form is suitable for lifted QCQP
formulations and subsequent semidefinite relaxations. If the semidefinite
relaxation is tight, meaning that the lifted solution has the required rank, the
relaxed problem recovers a globally optimal solution to the original QCQP.

In this notebook, we first solve the problem with GTSAM's nonlinear optimizer using an ordinary 3D point, then solve the same measurement problem in a QCQP formulation using `AugmentedPoint3`. The augmented type represents the **same 3D position**, but supplies the lifting traits that enable QCQP conversion. An optional SDP relaxation illustrates how to obtain a lower bound; simply using `AugmentedPoint3` does not make a local solve globally optimal.

Several geometrically distinct landmarks are required to uniquely determine the point. This example uses four known landmarks and noisy range measurements to estimate the 3D position.

See the `KnownLandmarkSquaredRangeFactor` [API notebook](../../../../gtsam/slam/doc/KnownLandmarkSquaredRangeFactor.ipynb) for more details on the factor.

GTSAM Copyright 2010-2026, Georgia Tech Research Corporation, Atlanta, Georgia 30332-0415
All Rights Reserved
Authors: Frank Dellaert, et al. (see THANKS for the full author list)
See LICENSE for the license information

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/borglab/gtsam/blob/develop/python/gtsam/examples/slam/KnownLandmarkSquaredRangeLocalizationExample.ipynb)

In [11]:
try:
    import google.colab
    %pip install --quiet gtsam-develop
except ImportError:
    pass

In [12]:
import numpy as np
import gtsam
import plotly.graph_objects as go


## Create a 3D synthetic localization problem

We choose one true 3D point and four fixed landmarks. The landmarks are spread through space so that the resulting range measurements provide independent geometric constraints.

Gaussian noise is added to each raw range measurement for realism.

In [13]:
rng = np.random.default_rng(42)

x_true = np.array([1.0, 2.0, 3.0])

landmarks = np.array([
    [0.0, 0.0, 0.0],
    [5.0, 0.0, 0.0],
    [0.0, 6.0, 0.0],
    [1.0, 0.0, 8.0],
])

range_sigma = 0.10  # raw range standard deviation [m]

true_ranges = [np.linalg.norm(x_true - wL) for wL in landmarks]

measurement_noise = rng.normal(
    loc=0.0, scale=range_sigma, size=len(landmarks)
)

measured_ranges = true_ranges + measurement_noise

print("True point:", x_true)
print(f"Raw range standard deviation: {range_sigma:.3f} m\n")

for i, (wL, r_true, r_measured) in enumerate(
    zip(landmarks, true_ranges, measured_ranges)):
    print(
        f"L{i}: {wL} "
        f"True range = {r_true:.3f} m, ",
        f"Measured range = {r_measured:.3f} m"
    )

True point: [1. 2. 3.]
Raw range standard deviation: 0.100 m

L0: [0. 0. 0.] True range = 3.742 m,  Measured range = 3.772 m
L1: [5. 0. 0.] True range = 5.385 m,  Measured range = 5.281 m
L2: [0. 6. 0.] True range = 5.099 m,  Measured range = 5.174 m
L3: [1. 0. 8.] True range = 5.385 m,  Measured range = 5.479 m


## Propagate range uncertainty into squared-range units

The sensor uncertainty above is expressed in range units, while `KnownLandmarkSquaredRangeFactor3` uses the squared-range residual

$$
e_i(x) = \|x - L_i\|^2 - d_i^2.
$$

For $y = d^2$, first-order uncertainty propagation gives

$$
\sigma_y \approx \left|\frac{\partial d^2}{\partial d}\right|\sigma_d = 2d\,\sigma_d.
$$

Therefore, the standard deviation used by each factor is approximately

$$
\sigma_{d^2} \approx 2d\,\sigma_d.
$$

Even when each range measurement has the same $\sigma_d$, their respective squared-range uncertainties differ between landmarks.

## Build the factor graph

Each range measurement contributes one factor to the same unknown point state.

In [14]:
key = 0
graph = gtsam.NonlinearFactorGraph()

for wL, measured_range in zip(landmarks, measured_ranges):
    sigma_squared_range = 2.0 * measured_range * range_sigma
    model = gtsam.noiseModel.Isotropic.Sigma(1, sigma_squared_range)
    graph.add(
        gtsam.KnownLandmarkSquaredRangeFactor3(key, wL, measured_range, model)
    )

## Choose an initial estimate

The nonlinear optimizer needs an initial value for the point state. The initial position is deliberately offset from the true position.

In [15]:
x_initial = np.array([1.2, 1.7, 3.2])

initial = gtsam.Values()
initial.insert(key, x_initial)

## Optimize

We solve the nonlinear least-squares problem with Levenberg-Marquardt.

In [16]:
optimizer = gtsam.LevenbergMarquardtOptimizer(graph, initial)
result = optimizer.optimize()

x_estimated = result.atVector(key)

print("True point: ", x_true)
print("Initial point: ", x_initial)
print("Estimated point: ", x_estimated)
print("Final graph error: ", graph.error(result))
print("Localization error: ", np.linalg.norm(x_estimated - x_true))

True point:  [1. 2. 3.]
Initial point:  [1.2 1.7 3.2]
Estimated point:  [1.14882058 1.99174407 2.96859758]
Final graph error:  0.41722601045710256
Localization error:  0.15232149470799192


## Convert and solve the QCQP

In C++, `KnownLandmarkSquaredRangeFactor<Position>` accepts either a fixed-size Eigen point (such as `Point3`) or `AugmentedPoint<N>`. In Python, `KnownLandmarkSquaredRangeFactor3` takes ordinary NumPy position coordinates, whereas `KnownLandmarkSquaredRangeFactorAugmentedPoint3` uses `gtsam.AugmentedPoint3`. **Both models estimate the same physical 3D point and have the same nonlinear residual.** Use the augmented representation specifically when you need the QCQP conversion interface.

`AugmentedPoint3` stores only three Euclidean coordinates; it does **not** store a five-dimensional point. During QCQP conversion its traits introduce $X=[h,x^\top,u]^\top$ with five entries. The canonical lift uses $h=1$ and $u=x^\top x$, while the QCQP constrains $x^\top x-hu=0$ and $h^2=1$.

We reuse the same landmarks, ranges, and noise models with the augmented factor. `QcqpProblem` constructs a quadratic objective with lifting constraints, and `insertQcqpValue3` initializes its lifted variables from the augmented point. The QCQP solver below remains a **local** optimizer; the optional SDP section provides a lower bound for comparison. These cells require a GTSAM build with augmented-point Python bindings.


In [17]:
augmented_graph = gtsam.NonlinearFactorGraph()
for wL, measured_range in zip(landmarks, measured_ranges):
    sigma_squared_range = 2.0 * measured_range * range_sigma
    model = gtsam.noiseModel.Isotropic.Sigma(1, sigma_squared_range)
    augmented_graph.add(
        gtsam.KnownLandmarkSquaredRangeFactorAugmentedPoint3(
            key, wL, measured_range, model
        )
    )

augmented_initial = gtsam.Values()
augmented_initial.insert(key, gtsam.AugmentedPoint3(x_initial))
qcqp = gtsam.QcqpProblem(augmented_graph, 1)
lifted_initial = gtsam.Values()
gtsam.insertQcqpValue3(
    key, augmented_initial.atAugmentedPoint3(key), lifted_initial
)
initial_cost, initial_equality, initial_inequality = qcqp.evaluate(lifted_initial)
np.testing.assert_allclose(initial_cost, graph.error(initial))
np.testing.assert_allclose([initial_equality, initial_inequality], 0.0, atol=1e-10)


We solve the QCQP using GTSAM's augmented-Lagrangian optimizer, then compare the recovered position and objective with Levenberg–Marquardt. This is a local solve; QCQP conversion alone does not certify a global optimum.

In [18]:
params = gtsam.AugmentedLagrangianParams()
params.maxIterations = 200
params.absoluteViolationTolerance = 1e-9
params.relativeViolationTolerance = 1e-9
params.relativeCostTolerance = 1e-12
params.absoluteStationarityTolerance = 1e-8
lifted_result = gtsam.AugmentedLagrangianOptimizer(
    qcqp, lifted_initial, params
).optimize()
recovered = gtsam.extractQcqpValues3(lifted_result)
x_qcqp = recovered.atAugmentedPoint3(key).vector()
qcqp_cost, equality_violation, inequality_violation = qcqp.evaluate(lifted_result)
feasible_cost = augmented_graph.error(recovered)
nonlinear_cost = graph.error(result)

print("QCQP position:", x_qcqp)
print("Position difference from LM:", np.linalg.norm(x_qcqp - x_estimated))
print(f"Objectives: LM={nonlinear_cost:.9f}, QCQP={qcqp_cost:.9f}")
print(f"Constraint violations: {equality_violation:.2e}, {inequality_violation:.2e}")
np.testing.assert_allclose(x_qcqp, x_estimated, rtol=0.0, atol=1e-4)
np.testing.assert_allclose(qcqp_cost, feasible_cost, rtol=1e-6, atol=1e-7)
np.testing.assert_allclose(feasible_cost, nonlinear_cost, rtol=1e-6, atol=1e-7)
assert max(equality_violation, inequality_violation) < 1e-6


QCQP position: [1.14880785 1.99173832 2.96859341]
Position difference from LM: 1.4578001410045574e-05
Objectives: LM=0.417226010, QCQP=0.417225902
Constraint violations: 6.37e-08, 0.00e+00


## Optional semidefinite relaxation

GTSAM's `MosekMonolithicSDP` backend can relax the QCQP to obtain a lower bound. If a feasible recovered point attains that bound, the solution is globally optimal; the relaxation is not necessarily tight.

We fix $h=1$ to remove the homogeneous sign ambiguity before relaxation. This section requires a GTSAM build with MOSEK support and a usable license.

In [19]:
sdp_position = None
if not hasattr(gtsam, "MosekMonolithicSDP"):
    print("This GTSAM build does not include the optional MOSEK backend.")
else:
    sdp_qcqp = gtsam.QcqpProblem(augmented_graph, 1)
    h_selector = np.zeros((1, 5))
    h_selector[0, 0] = 1.0
    sdp_qcqp.addConstraint(gtsam.LinearConstraint.Equal(gtsam.JacobianFactor(
        key, h_selector, np.ones(1), gtsam.noiseModel.Unit.Create(1)
    )))
    sdp = gtsam.MosekMonolithicSDP(sdp_qcqp)
    if not sdp.solve({"intpntCoTolRelGap": 1e-10}):
        raise RuntimeError("The SDP solver did not return a readable solution")
    status = sdp.problemStatus()
    print("SDP problem status:", status)
    if status != "ProblemStatus::PrimalAndDualFeasible":
        raise RuntimeError(f"Cannot interpret this SDP result as an optimum: {status}")
    sdp_recovered = gtsam.extractQcqpValues3(sdp.qcqpValues())
    sdp_position = sdp_recovered.atAugmentedPoint3(key).vector()
    lower_bound = sdp.objectiveValue()
    sdp_feasible_cost = augmented_graph.error(sdp_recovered)
    gap = sdp_feasible_cost - lower_bound
    tolerance = 1e-6 * max(1.0, abs(nonlinear_cost))
    print("SDP position:", sdp_position)
    print(f"SDP lower bound: {lower_bound:.9f}")
    print(f"Recovered nonlinear cost: {sdp_feasible_cost:.9f}, gap: {gap:.2e}")
    print("Position difference from LM:", np.linalg.norm(sdp_position - x_estimated))
    print("Eigenvalue ratios:", sdp.variableEVRs())
    assert lower_bound <= nonlinear_cost + tolerance
    if abs(gap) <= tolerance:
        print("The recovered point matches the SDP bound within tolerance.")
    else:
        print("The recovered point is not certified by this SDP bound.")


SDP problem status: ProblemStatus::PrimalAndDualFeasible
SDP position: [1.14880786 1.99173845 2.96859343]
SDP lower bound: 0.417226004
Recovered nonlinear cost: 0.417226004, gap: -2.86e-11
Position difference from LM: 1.4513146075643321e-05
Eigenvalue ratios: [813455052060502.4]
The recovered point matches the SDP bound within tolerance.


## Visualize the result

The interactive 3D plot below shows the known landmarks, true position,
initial estimate, and optimized estimate. Rotate or zoom the plot to inspect
the geometry from different viewpoints. The QCQP estimate is also shown, along
with the SDP estimate if available.

In [20]:
fig = go.Figure()

fig.add_scatter3d(x=landmarks[:, 0], y=landmarks[:, 1], z=landmarks[:, 2], mode="markers", marker=dict(size=6), name="Known Landmarks",
                  hovertemplate="Landmark<br>x=%{x:.2f}<br>y=%{y:.2f}<br>z=%{z:.2f}<extra></extra>")
fig.add_scatter3d(x=[x_true[0]], y=[x_true[1]], z=[x_true[2]], mode="markers", marker=dict(size=7), name="True Position")
fig.add_scatter3d(x=[x_initial[0]], y=[x_initial[1]], z=[x_initial[2]], mode="markers", marker=dict(size=7), name="Initial Estimate")
fig.add_scatter3d(x=[x_estimated[0]], y=[x_estimated[1]], z=[x_estimated[2]], mode="markers", marker=dict(size=7), name="Estimated Position")
fig.add_scatter3d(x=[x_qcqp[0]], y=[x_qcqp[1]], z=[x_qcqp[2]], mode="markers", marker=dict(size=7, symbol="diamond"), name="QCQP (ALM)")

if sdp_position is not None:
    fig.add_scatter3d(x=[sdp_position[0]], y=[sdp_position[1]], z=[sdp_position[2]], mode="markers", marker=dict(size=7, symbol="cross"), name="SDP")

fig.add_scatter3d(x=[x_true[0], x_estimated[0]], y=[x_true[1], x_estimated[1]], z=[x_true[2], x_estimated[2]], mode="lines", name="Estimation Error")

fig.update_layout(
    title="3D Squared-Range Localization", width=800, height=650,
    scene=dict(xaxis_title="x", yaxis_title="y", zaxis_title="z", aspectmode="data"),
    legend=dict(x=0.02, y=0.98), margin=dict(l=0, r=0, b=0, t=50),
)
fig.show()

## Takeaways

- `KnownLandmarkSquaredRangeFactor3` estimates a 3D Euclidean point, not a pose.
- Each factor relates the point to one fixed known landmark.
- Pass the non-negative range measurement $d$ to the constructor; the factor stores $d^2$ internally.
- The residual is scalar, so each factor uses a one-dimensional noise model.
- Raw range uncertainty should be propagated into squared-range units before use in the factor noise model.
- Known landmarks are fixed factor data and are not inserted into `Values`.
- `AugmentedPoint3` represents the same three-dimensional position as an ordinary point; its QCQP traits construct a five-dimensional lift only during conversion.
- `KnownLandmarkSquaredRangeFactorAugmentedPoint3` provides a QCQP formulation of the same objective; the local QCQP solution can be checked against an SDP lower bound when available.